In [ ]:
# Cell 1 — Imports
import json
import pandas as pd
import matplotlib.pyplot as plt
import os

RESULTS_DIR = "../results"
PLOTS_DIR   = "../plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
# Load CSVs
memory_df     = pd.read_csv(f"{RESULTS_DIR}/memory_results.csv")
latency_df    = pd.read_csv(f"{RESULTS_DIR}/latency_results.csv")
throughput_df = pd.read_csv(f"{RESULTS_DIR}/throughput_results.csv")

print("Memory:\n", memory_df)
print("\nLatency:\n", latency_df.head())
print("\nThroughput:\n", throughput_df.head())

In [ ]:
# Memory bar chart
# Three grouped bars per model: allocated, reserved, peak
models   = memory_df.index.tolist()
x        = range(len(models))
width    = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar([i - width for i in x], memory_df["allocated_gb"],  width, label="Weights (allocated)")
ax.bar([i          for i in x], memory_df["reserved_gb"],   width, label="Idle (reserved)")
ax.bar([i + width for i in x], memory_df["peak_2k_gb"],    width, label="Peak at 2K ctx")

ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylabel("GPU Memory (GB)")
ax.set_title("Memory Footprint by Quantization Variant")
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/memory_comparison.png", dpi=150)
plt.show()
print("Saved memory_comparison.png")

In [ ]:
# Prefill latency line chart
# One line per model, x = prompt length, y = prefill time
prefill_df = latency_df[latency_df["phase"] == "prefill"]

fig, ax = plt.subplots(figsize=(10, 6))
for model_name in prefill_df["model"].unique():
    subset = prefill_df[prefill_df["model"] == model_name]
    ax.plot(subset["prompt_length"], subset["time_sec"],
            marker="o", label=model_name)

ax.set_xlabel("Prompt Length (tokens)")
ax.set_ylabel("Prefill Time (sec)")
ax.set_title("Prefill Latency vs Prompt Length")
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/prefill_latency.png", dpi=150)
plt.show()
print("Saved prefill_latency.png")

In [ ]:
# Decode latency line chart
# One line per model, x = prompt length, y = per-token decode time
decode_df = latency_df[latency_df["phase"] == "decode"]

fig, ax = plt.subplots(figsize=(10, 6))
for model_name in decode_df["model"].unique():
    subset = decode_df[decode_df["model"] == model_name]
    ax.plot(subset["prompt_length"], subset["time_sec"],
            marker="o", label=model_name)

ax.set_xlabel("KV Cache Length (tokens)")
ax.set_ylabel("Per-Token Decode Time (sec)")
ax.set_title("Decode Latency vs Context Length")
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/decode_latency.png", dpi=150)
plt.show()
print("Saved decode_latency.png")

In [ ]:
# TTFT line chart
# Time to first token — one line per model, x = prompt length
ttft_df = latency_df[latency_df["phase"] == "ttft"]

fig, ax = plt.subplots(figsize=(10, 6))
for model_name in ttft_df["model"].unique():
    subset = ttft_df[ttft_df["model"] == model_name]
    ax.plot(subset["prompt_length"], subset["time_sec"],
            marker="o", label=model_name)

ax.set_xlabel("Prompt Length (tokens)")
ax.set_ylabel("Time to First Token (sec)")
ax.set_title("TTFT vs Prompt Length")
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/ttft.png", dpi=150)
plt.show()
print("Saved ttft.png")

In [ ]:
# Throughput bar chart
# Grouped bars: batch size 1, 4, 16 for each model
tput_df = throughput_df[throughput_df["phase"] == "throughput"]

fig, ax = plt.subplots(figsize=(10, 6))
batch_sizes = sorted(tput_df["batch_size"].unique())
models      = tput_df["model"].unique()
width       = 0.2
x           = range(len(batch_sizes))

for i, model_name in enumerate(models):
    subset = tput_df[tput_df["model"] == model_name]
    subset = subset.sort_values("batch_size")
    ax.bar([j + i * width for j in x],
           subset["tokens_per_sec"],
           width, label=model_name)

ax.set_xticks([j + width for j in x])
ax.set_xticklabels([f"Batch {bs}" for bs in batch_sizes])
ax.set_ylabel("Tokens / sec")
ax.set_title("Throughput by Batch Size")
ax.legend()
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/throughput.png", dpi=150)
plt.show()
print("Saved throughput.png")

In [ ]:
#OOM sweep table
# Shows max batch size each variant can handle at 2K context
oom_df = throughput_df[throughput_df["phase"] == "oom_sweep"]
print("\nOOM Sweep Results (2K context):")
print(oom_df[["model", "batch_size", "status"]].to_string(index=False))